In [3]:
import os
print(os.getcwd())

/Users/guypigott/python-venv-demo


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

# =========================================================
# PATHS
# =========================================================

BASE_DIR = Path.cwd()   # current working directory
DATA_DIR = BASE_DIR / "EDS" / "Data" / "Output"

INPUT_FILE = DATA_DIR / "acs_ssc_analysis_pre_lasso.pkl"
OUTPUT_FILE = DATA_DIR / "acs_ssc_predicted_merged.pkl"

# =========================================================
# SETTINGS
# =========================================================
TRAIN_YEAR = 2012
CV = 10
MAX_ITER = 10000
MIN_INTERACTION_SUPPORT = 10
RANDOM_STATE = 42

# =========================================================
# HELPERS
# =========================================================

def check_required_columns(df: pd.DataFrame, required: list[str]) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def build_pooled_individual_data(df_couple: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """
    Build pooled person-level dataset:
    first n_couples rows = respondent/head-like side
    next n_couples rows  = spouse/partner side
    """
    base_r = df_couple[
        [
            "year",
            "statefip",
            "c_children",
            "r_incearn",
            "r_agegroup",
            "r_edugroup",
            "r_race",
            "r_occbroad",
            "r_degbroad",
        ]
    ].rename(
        columns={
            "r_incearn": "incearn",
            "r_agegroup": "agegroup",
            "r_edugroup": "edugroup",
            "r_race": "race",
            "r_occbroad": "occbroad",
            "r_degbroad": "degbroad",
        }
    )

    base_sp = df_couple[
        [
            "year",
            "statefip",
            "c_children",
            "sp_incearn",
            "sp_agegroup",
            "sp_edugroup",
            "sp_race",
            "sp_occbroad",
            "sp_degbroad",
        ]
    ].rename(
        columns={
            "sp_incearn": "incearn",
            "sp_agegroup": "agegroup",
            "sp_edugroup": "edugroup",
            "sp_race": "race",
            "sp_occbroad": "occbroad",
            "sp_degbroad": "degbroad",
        }
    )

    n_couples = len(df_couple)
    df_indiv = pd.concat([base_r, base_sp], axis=0, ignore_index=True)
    return df_indiv, n_couples


def build_feature_matrix(df_indiv: pd.DataFrame) -> pd.DataFrame:
    """
    Lighter pooled LASSO feature matrix.

    Keeps:
    - main effects for agegroup, education, race, occupation, degree, state, year
    - c_children as numeric
    - structured interactions centered on age, education, children, and year

    This is closer to the Stata setup than full pairwise interactions.
    """
    cat_cols = [
        "agegroup",
        "edugroup",
        "race",
        "occbroad",
        "degbroad",
        "statefip",
        "year",
    ]
    num_cols = ["c_children"]

    X = pd.get_dummies(
        df_indiv[cat_cols + num_cols],
        columns=cat_cols,
        drop_first=True,
        dtype=float,
    )

    cols = list(X.columns)
    interaction_parts = []

    def add_interactions(left_prefixes, right_prefixes, min_support=10):
        left_cols = [c for c in cols if any(c.startswith(p) for p in left_prefixes)]
        right_cols = [c for c in cols if any(c.startswith(p) for p in right_prefixes)]

        for c1 in left_cols:
            v1 = X[c1].to_numpy()
            for c2 in right_cols:
                if c1 == c2:
                    continue
                # avoid duplicates like A×B and B×A
                if c1 > c2:
                    continue
                s = v1 * X[c2].to_numpy()
                if np.nansum(s) >= min_support:
                    interaction_parts.append(pd.Series(s, name=f"{c1}_x_{c2}"))

    # Interactions chosen to mimic the spirit of the paper
    add_interactions(["agegroup_"], ["edugroup_"])
    add_interactions(["agegroup_"], ["race_"])
    add_interactions(["agegroup_"], ["occbroad_"])
    add_interactions(["agegroup_"], ["degbroad_"])
    add_interactions(["agegroup_"], ["statefip_"])
    add_interactions(["agegroup_"], ["year_"])

    # Children with key dimensions
    add_interactions(["c_children"], ["agegroup_"])
    add_interactions(["c_children"], ["edugroup_"])

    if interaction_parts:
        X = pd.concat([X] + interaction_parts, axis=1)

    return X.astype(float)


def fit_two_step_lasso(
    X_df: pd.DataFrame,
    y: np.ndarray,
    train_mask: np.ndarray,
) -> tuple[LassoCV, LassoCV, StandardScaler]:
    """
    Step 1: Lasso for positive earnings
    Step 2: Lasso for earnings conditional on positive earnings
    """
    scaler = StandardScaler()
    X_train = X_df.loc[train_mask]
    scaler.fit(X_train)

    X_sc = scaler.transform(X_df)
    X_train_sc = X_sc[train_mask]

    y_pos = (y > 0).astype(float)
    y_pos_train = y_pos[train_mask]

    m1 = LassoCV(
        cv=CV,
        max_iter=MAX_ITER,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    m1.fit(X_train_sc, y_pos_train)

    pos_train_mask = train_mask & (y > 0)
    if pos_train_mask.sum() == 0:
        raise ValueError("No positive earners in the training sample for step 2.")

    X_pos_train_sc = X_sc[pos_train_mask]
    y_pos_level = y[pos_train_mask]

    m2 = LassoCV(
        cv=CV,
        max_iter=MAX_ITER,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    m2.fit(X_pos_train_sc, y_pos_level)

    return m1, m2, scaler


def predict_two_step(
    X_df: pd.DataFrame,
    y: np.ndarray,
    train_mask: np.ndarray,
    m1: LassoCV,
    m2: LassoCV,
    scaler: StandardScaler,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    """
    Returns:
      predicted_probability_positive,
      predicted_binary_positive,
      predicted_earnings,
      threshold
    """
    X_sc = scaler.transform(X_df)

    # Step 1: probability-like LPM prediction
    hat_pos_prob = m1.predict(X_sc)

    # Match overall positive-earnings share in the cleaned pooled sample
    mean_pos_actual = (y > 0).mean()
    threshold = np.quantile(hat_pos_prob, 1 - mean_pos_actual)
    hat_pos = (hat_pos_prob >= threshold).astype(float)

    # Step 2: positive earnings level
    hat_earn_level = np.maximum(m2.predict(X_sc), 0.0)

    # Final combined prediction
    hat_incearn = np.where(hat_pos == 1, hat_earn_level, 0.0)

    return hat_pos_prob, hat_pos, hat_incearn, threshold


def add_couple_predictions(
    df_couple: pd.DataFrame,
    n_couples: int,
    hat_pos_prob_all: np.ndarray,
    hat_pos_all: np.ndarray,
    hat_incearn_all: np.ndarray,
) -> pd.DataFrame:
    df = df_couple.copy()

    # first block = respondent, second block = spouse
    df["hat_pos_prob_r"] = hat_pos_prob_all[:n_couples]
    df["hat_pos_r"] = hat_pos_all[:n_couples]
    df["hat_incearn_r"] = hat_incearn_all[:n_couples]

    df["hat_pos_prob_sp"] = hat_pos_prob_all[n_couples:]
    df["hat_pos_sp"] = hat_pos_all[n_couples:]
    df["hat_incearn_sp"] = hat_incearn_all[n_couples:]

    df["hat_c_incearn"] = df["hat_incearn_r"] + df["hat_incearn_sp"]

    total = df["hat_c_incearn"].replace(0, np.nan)
    df["hat_c_split"] = (
        np.maximum(df["hat_incearn_r"], df["hat_incearn_sp"]) / total
    ).fillna(0.5)

    for i in range(1, 6):
        df[f"hat_c_incearn{i}"] = (df["hat_c_incearn"] / 10000) ** i
        df[f"hat_c_split{i}"] = df["hat_c_split"] ** i

    return df


def print_diagnostics(df: pd.DataFrame, train_year: int, threshold: float) -> None:
    print("=" * 70)
    print("MERGED LASSO DIAGNOSTICS")
    print("=" * 70)
    print(f"Threshold used for positive earnings: {threshold:.6f}")
    print(f"Training year: {train_year}")
    print()

    print("--- Sample counts ---")
    print(f"Total couples: {len(df):,}")
    print(f"Married couples: {int(df['sscouple_mar'].sum()):,}")
    print(f"Cohabiting couples: {int(df['sscouple_coh'].sum()):,}")
    print()

    print("--- Share positive earnings ---")
    print(
        f"Respondent  reported={(df['r_incearn'] > 0).mean():.3f}  "
        f"predicted={df['hat_pos_r'].mean():.3f}"
    )
    print(
        f"Spouse      reported={(df['sp_incearn'] > 0).mean():.3f}  "
        f"predicted={df['hat_pos_sp'].mean():.3f}"
    )
    print()

    print("--- Couple predicted total earnings ---")
    for label, mask in [
        ("Married", df["sscouple_mar"] == 1),
        ("Cohabiting", df["sscouple_coh"] == 1),
    ]:
        sub = df.loc[mask]
        print(
            f"{label:<11}: mean={sub['hat_c_incearn'].mean():10,.0f}  "
            f"sd={sub['hat_c_incearn'].std():10,.0f}"
        )
    print()

    print("--- Couple predicted earnings split ---")
    for label, mask in [
        ("Married", df["sscouple_mar"] == 1),
        ("Cohabiting", df["sscouple_coh"] == 1),
    ]:
        sub = df.loc[mask]
        pos = sub["hat_c_incearn"] > 0
        print(
            f"{label:<11}: mean={sub.loc[pos, 'hat_c_split'].mean():.3f}  "
            f"sd={sub.loc[pos, 'hat_c_split'].std():.3f}"
        )
    print()

    print("--- Correlation reported vs predicted (positive earners) ---")
    for pfx, label in [("r", "Respondent"), ("sp", "Spouse")]:
        both = (df[f"{pfx}_incearn"] > 0) & (df[f"hat_incearn_{pfx}"] > 0)
        if both.sum() > 2:
            corr = np.corrcoef(
                df.loc[both, f"{pfx}_incearn"],
                df.loc[both, f"hat_incearn_{pfx}"],
            )[0, 1]
            print(f"{label:<11}: r={corr:.3f}  n={int(both.sum()):,}")
        else:
            print(f"{label:<11}: insufficient positive-earner overlap")
    print()

    print("--- Paper-style reference targets ---")
    print("Predicted positive earnings: married ≈ 0.963, cohabiting ≈ 0.969")
    print("Predicted couple earnings:   married ≈ 110,729, cohabiting ≈ 102,953")
    print("Predicted earnings split:    married ≈ 0.648, cohabiting ≈ 0.641")
    print()

    print("--- By marital status: predicted positive earnings share ---")
    for label, mask in [
        ("Married", df["sscouple_mar"] == 1),
        ("Cohabiting", df["sscouple_coh"] == 1),
    ]:
        sub = df.loc[mask]
        # paper Table 2 reports couple-level positive earnings, so show both-partner based share too
        couple_pos = ((sub["hat_incearn_r"] > 0) | (sub["hat_incearn_sp"] > 0)).mean()
        both_pos = ((sub["hat_incearn_r"] > 0) & (sub["hat_incearn_sp"] > 0)).mean()
        print(
            f"{label:<11}: any positive={couple_pos:.3f}  both positive={both_pos:.3f}"
        )


def main() -> None:
    print(f"Loading pre-LASSO analysis sample from:\n{INPUT_FILE}")
    df_couple = pd.read_pickle(INPUT_FILE)

    required_cols = [
        "year",
        "statefip",
        "sscouple_mar",
        "sscouple_coh",
        "age",
        "sp_age",
        "r_incearn",
        "sp_incearn",
        "r_agegroup",
        "sp_agegroup",
        "r_edugroup",
        "sp_edugroup",
        "r_race",
        "sp_race",
        "r_occbroad",
        "sp_occbroad",
        "r_degbroad",
        "sp_degbroad",
        "c_children",
        "c_agemin",
        "c_agemax",
        "r_earnstatus",
    ]
    check_required_columns(df_couple, required_cols)

    # Keep the richer build's paper-style pre-LASSO restrictions
    df_couple = df_couple[
        (df_couple["c_agemin"] >= 18) &
        (df_couple["c_agemax"] <= 60) &
        (df_couple["r_earnstatus"] == 1)
    ].copy()

    print(f"Loaded couple sample: {len(df_couple):,}")
    print(f"Married share: {df_couple['sscouple_mar'].mean():.3f}")
    print(f"Share with children: {df_couple['c_anychildren'].mean():.3f}" if "c_anychildren" in df_couple.columns else "")

    # Build pooled individual data
    df_indiv, n_couples = build_pooled_individual_data(df_couple)
    print(f"Pooled individual rows: {len(df_indiv):,}")
    print(f"Training rows in {TRAIN_YEAR}: {(df_indiv['year'] == TRAIN_YEAR).sum():,}")

    # Build features
    X_df = build_feature_matrix(df_indiv)
    y = df_indiv["incearn"].to_numpy()
    train_mask = (df_indiv["year"].to_numpy() == TRAIN_YEAR)

    print(f"Feature matrix shape: {X_df.shape}")
    print("Fitting two-step pooled LASSO...")

    # Fit and predict
    m1, m2, scaler = fit_two_step_lasso(X_df, y, train_mask)
    hat_pos_prob_all, hat_pos_all, hat_incearn_all, threshold = predict_two_step(
        X_df, y, train_mask, m1, m2, scaler
    )

    # Add predictions back to couple file
    df_out = add_couple_predictions(
        df_couple,
        n_couples,
        hat_pos_prob_all,
        hat_pos_all,
        hat_incearn_all,
    )

    # Diagnostics
    print_diagnostics(df_out, TRAIN_YEAR, threshold)

    # Save
    df_out.to_pickle(OUTPUT_FILE)
    print()
    print(f"Saved merged LASSO output to:\n{OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Loading pre-LASSO analysis sample from:
/Users/guypigott/python-venv-demo/EDS/Data/Output/acs_ssc_analysis_pre_lasso.pkl
Loaded couple sample: 38,279
Married share: 0.422
Share with children: 0.240
Pooled individual rows: 76,558
Training rows in 2012: 10,404
Feature matrix shape: (76558, 883)
Fitting two-step pooled LASSO...
MERGED LASSO DIAGNOSTICS
Threshold used for positive earnings: 0.811216
Training year: 2012

--- Sample counts ---
Total couples: 38,279
Married couples: 16,147
Cohabiting couples: 22,132

--- Share positive earnings ---
Respondent  reported=0.970  predicted=0.926
Spouse      reported=0.794  predicted=0.839

--- Couple predicted total earnings ---
Married    : mean=   110,203  sd=    55,499
Cohabiting : mean=   101,409  sd=    52,153

--- Couple predicted earnings split ---
Married    : mean=0.665  sd=0.175
Cohabiting : mean=0.656  sd=0.164

--- Correlation reported vs predicted (positive earners) ---
Respondent : r=0.482  n=35,061
Spouse     : r=0.445  n=28,749

-

In [9]:
import pandas as pd

df_out = pd.read_pickle(OUTPUT_FILE)

print(df_out['hat_c_incearn'].describe())
print(df_out['hat_c_split'].describe())

count     38279.000000
mean     105118.280836
std       53764.698970
min           0.000000
25%       66252.640819
50%      100931.683330
75%      142959.379096
max      554864.666430
Name: hat_c_incearn, dtype: float64
count    38279.000000
mean         0.654435
std          0.167956
min          0.500000
25%          0.531456
50%          0.591518
75%          0.696015
max          1.000000
Name: hat_c_split, dtype: float64


In [ ]:
import pandas as pd

df_out = pd.read_pickle(r"C:\Users\asus\Desktop\acs_ssc_predicted_merged.pkl")

print(df_out['hat_c_incearn'].describe())
print(df_out['hat_c_split'].describe())